In [2]:
import pandas as pd
from math import sqrt
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

In [3]:
movies_df=pd.read_csv(r"C:\Users\Acer\Desktop\Data_Science_Projects\Recommender system\Content_based\data\movies.csv")
ratings_df=pd.read_csv(r"C:\Users\Acer\Desktop\Data_Science_Projects\Recommender system\Content_based\data\ratings.csv")
movies_df.head()


,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [4]:
ratings_df.head()

,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


In [5]:
#preprocessing
#extract year
movies_df['year']=movies_df.title.str.extract('(\(\d\d\d\d\))',expand=False)
movies_df['year']=movies_df.year.str.extract('(\d\d\d\d)',expand=False)
movies_df['title']=movies_df.title.str.replace('(\(\d\d\d\d\))',"",regex=True)
movies_df['title']=movies_df['title'].apply(lambda x:x.strip())
movies_df.head()

,movieId,title,genres,year
0,1,Toy Story,Adventure|Animation|Children|Comedy|Fantasy,1995
1,2,Jumanji,Adventure|Children|Fantasy,1995
2,3,Grumpier Old Men,Comedy|Romance,1995
3,4,Waiting to Exhale,Comedy|Drama|Romance,1995
4,5,Father of the Bride Part II,Comedy,1995


In [6]:
#Split genres
movies_df['genres']=movies_df.genres.str.split('|')
movies_df.head()

,movieId,title,genres,year
0,1,Toy Story,"[Adventure, Animation, Children, Comedy, Fantasy]",1995
1,2,Jumanji,"[Adventure, Children, Fantasy]",1995
2,3,Grumpier Old Men,"[Comedy, Romance]",1995
3,4,Waiting to Exhale,"[Comedy, Drama, Romance]",1995
4,5,Father of the Bride Part II,[Comedy],1995


In [7]:
#One-Hot Encoding Genres
moviesWithGenres_df=movies_df.copy()
for index,row in movies_df.iterrows():
    for genre in row['genres']:
        moviesWithGenres_df.at[index,genre]=1
moviesWithGenres_df=moviesWithGenres_df.fillna(0)
moviesWithGenres_df.head()

,movieId,title,genres,year,Adventure,Animation,Children,Comedy,Fantasy,Romance,...,Horror,Mystery,Sci-Fi,War,Musical,Documentary,IMAX,Western,Film-Noir,(no genres listed)
0,1,Toy Story,"[Adventure, Animation, Children, Comedy, Fantasy]",1995,1.0,1.0,1.0,1.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2,Jumanji,"[Adventure, Children, Fantasy]",1995,1.0,0.0,1.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,3,Grumpier Old Men,"[Comedy, Romance]",1995,0.0,0.0,0.0,1.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,4,Waiting to Exhale,"[Comedy, Drama, Romance]",1995,0.0,0.0,0.0,1.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,5,Father of the Bride Part II,[Comedy],1995,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [8]:
import random
userId=random.choice(ratings_df['userId'].unique())
print("Selected User :",userId)

Selected User : 136


In [9]:
userSubset=ratings_df[ratings_df['userId']==userId]
userSubset.head()

,userId,movieId,rating,timestamp
20615,136,10,5.0,832449222
20616,136,15,5.0,832449934
20617,136,16,5.0,832449614
20618,136,19,3.0,832449345
20619,136,23,5.0,832449838


In [10]:
#Merge with movies dataset
userSubset=pd.merge(userSubset,movies_df,on='movieId')
userSubset.head()

,userId,movieId,rating,timestamp,title,genres,year
0,136,10,5.0,832449222,GoldenEye,"[Action, Adventure, Thriller]",1995
1,136,15,5.0,832449934,Cutthroat Island,"[Action, Adventure, Romance]",1995
2,136,16,5.0,832449614,Casino,"[Crime, Drama]",1995
3,136,19,3.0,832449345,Ace Ventura: When Nature Calls,[Comedy],1995
4,136,23,5.0,832449838,Assassins,"[Action, Crime, Thriller]",1995


In [11]:
#Extract Genre Matrix for user movies
userMovies=moviesWithGenres_df[moviesWithGenres_df['movieId'].isin(userSubset['movieId'].tolist())]
userMovies=userMovies.reset_index(drop=True)
userGenreTable=userMovies.drop(['movieId','title','genres','year'],axis=1)
userGenreTable.head()

,Adventure,Animation,Children,Comedy,Fantasy,Romance,Drama,Action,Crime,Thriller,Horror,Mystery,Sci-Fi,War,Musical,Documentary,IMAX,Western,Film-Noir,(no genres listed)
0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [12]:
#Build User Profile
userProfile=userGenreTable.transpose().dot(userSubset['rating'])
userProfile

Adventure              73.0
Animation              15.0
Children               27.0
Comedy                136.0
Fantasy                31.0
Romance                68.0
Drama                 139.0
Action                126.0
Crime                  65.0
Thriller              108.0
Horror                 28.0
Mystery                21.0
Sci-Fi                 42.0
War                    24.0
Musical                14.0
Documentary             0.0
IMAX                   10.0
Western                15.0
Film-Noir               0.0
(no genres listed)      0.0
dtype: float64

In [13]:
#Prepare Full Genre Table
genreTable=moviesWithGenres_df.set_index(moviesWithGenres_df['movieId'])
genreTable=genreTable.drop(['movieId','title','genres','year'],axis=1)

In [14]:
#compute Recommendation Scores
recommendationTable_df=((genreTable*userProfile).sum(axis=1))/(userProfile.sum())
recommendationTable_df.head()

movieId
1    0.299363
2    0.139066
3    0.216561
4    0.364119
5    0.144374
dtype: float64

In [15]:
#Sort and Recommend
recommendationTable_df=recommendationTable_df.sort_values(ascending=False)
recommendedMovies=movies_df.loc[movies_df['movieId'].isin(recommendationTable_df.head(10).keys())]
recommendedMovies[['title','genres']]

,title,genres
400,"Getaway, The","[Action, Adventure, Crime, Drama, Romance, Thr..."
3460,Osmosis Jones,"[Action, Animation, Comedy, Crime, Drama, Roma..."
3608,"Stunt Man, The","[Action, Adventure, Comedy, Drama, Romance, Th..."
4693,"Last Boy Scout, The","[Action, Comedy, Crime, Drama, Thriller]"
4843,Ichi the Killer (Koroshiya 1),"[Action, Comedy, Crime, Drama, Horror, Thriller]"
5774,"Chase, The","[Action, Adventure, Comedy, Crime, Romance, Th..."
6570,"Hunting Party, The","[Action, Adventure, Comedy, Drama, Thriller]"
7170,Aelita: The Queen of Mars (Aelita),"[Action, Adventure, Drama, Fantasy, Romance, S..."
7441,Rubber,"[Action, Adventure, Comedy, Crime, Drama, Film..."
8597,Dragonheart 2: A New Beginning,"[Action, Adventure, Comedy, Drama, Fantasy, Th..."


In [16]:
#Remoce Already watched movies
alreadyWatched=userSubset['movieId'].tolist()
recommendationTable_df=recommendationTable_df.drop(alreadyWatched,errors='ignore')
#sort again
recommendationTable_df=recommendationTable_df.sort_values(ascending=False)
#show final top 10
top10=recommendationTable_df.head(10)
recommendedMovies=movies_df[movies_df['movieId'].isin(top10.index)]
recommendedMovies[['title','genres']]

,title,genres
19,Money Train,"[Action, Comedy, Crime, Drama, Thriller]"
400,"Getaway, The","[Action, Adventure, Crime, Drama, Romance, Thr..."
3460,Osmosis Jones,"[Action, Animation, Comedy, Crime, Drama, Roma..."
3608,"Stunt Man, The","[Action, Adventure, Comedy, Drama, Romance, Th..."
4843,Ichi the Killer (Koroshiya 1),"[Action, Comedy, Crime, Drama, Horror, Thriller]"
5774,"Chase, The","[Action, Adventure, Comedy, Crime, Romance, Th..."
6570,"Hunting Party, The","[Action, Adventure, Comedy, Drama, Thriller]"
7170,Aelita: The Queen of Mars (Aelita),"[Action, Adventure, Drama, Fantasy, Romance, S..."
7441,Rubber,"[Action, Adventure, Comedy, Crime, Drama, Film..."
8597,Dragonheart 2: A New Beginning,"[Action, Adventure, Comedy, Drama, Fantasy, Th..."
